In [1]:
import pandas as pd
import os
import glob
from IPython.display import display

In [2]:


# Directory paths
input_dir = '../02_Processed_Data/01_Sep_Cutting_Lines'
output_dir = '../02_Processed_Data/02_Necessary_Columns'

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Find files in the 01_Sep_Cutting directory
file_paths = glob.glob(os.path.join(input_dir, '*.csv'))
summary_data = []

# Column finder helper function (Removed case sensitivity)
def find_column(df, search_word):
    for col in df.columns:
        if search_word.lower() in str(col).lower():
            return col
    return None

for file_path in file_paths:
    file_name = os.path.basename(file_path)
    
    # ==========================================
    # 1. EXTRACTING FILE ID
    # ==========================================
    # Get the part after the '-' sign in the file name, discard the '.csv' part
    # E.g.: "cutting_data-1.csv" -> "1" or "cutting_test-A.csv" -> "A"
    try:
        file_id = file_name.split('-')[-1].replace('.csv', '')
    except Exception:
        file_id = "Unknown"
        
    # Read the cut data
    df = pd.read_csv(file_path)
    
    # ==========================================
    # 2. FINDING RELEVANT COLUMNS
    # ==========================================
    col_rawtime = find_column(df, "RawTime")
    col_imotor = find_column(df, "Machine.IMotor")
    col_trq = find_column(df, "Kafa.Act.Trq")
    col_x = find_column(df, "X_Accelerometer")
    col_y = find_column(df, "Y_Accelerometer")
    col_z = find_column(df, "Z_Accelerometer")
    
    # Create the new dataframe
    df_new = pd.DataFrame()
    
    # Fill in the data according to the desired column order
    df_new['File_ID'] = [file_id] * len(df) # Assign the extracted ID to all rows
    df_new['RawTime'] = df[col_rawtime] if col_rawtime else None
    df_new['IMotor'] = df[col_imotor] if col_imotor else None
    df_new['Kafa_Act_Trq'] = df[col_trq] if col_trq else None
    df_new['X_Accelerometer'] = df[col_x] if col_x else None
    df_new['Y_Accelerometer'] = df[col_y] if col_y else None
    df_new['Z_Accelerometer'] = df[col_z] if col_z else None
    
    # ==========================================
    # 3. DATA TYPE CLEANING (NUMERIZATION)
    # ==========================================
    # Ensure that sensor data such as accelerometer and motor current are completely float for model training.
    numeric_columns = ['RawTime', 'IMotor', 'Kafa_Act_Trq', 'X_Accelerometer', 'Y_Accelerometer', 'Z_Accelerometer']
    for col in numeric_columns:
        if df_new[col].dtype == object: # If it is a string with a comma in it
            df_new[col] = df_new[col].astype(str).str.replace(',', '.').apply(pd.to_numeric, errors='coerce')

    # ==========================================
    # 4. SAVING THE NEW FILE
    # ==========================================
    # Put the "nec_cols_" prefix instead of "cutting_" to keep track of the file name
    saved_filename = file_name.replace('cutting_', 'nec_cols_')
    processed_file_path = os.path.join(output_dir, saved_filename)
    
    df_new.to_csv(processed_file_path, index=False)
    
    # Record for the summary table
    summary_data.append({
        'Original File': file_name,
        'File ID': file_id,
        'Number of Columns Extracted': df_new.shape[1],
        'Number of Rows': len(df_new),
        'Status': 'Successful'
    })

# Print Results as a Table
summary_df = pd.DataFrame(summary_data)
display(summary_df)

print(f"✅ The required columns have been extracted, and the data has been numerized and saved in the '{output_dir}' directory.")

,Original File,File ID,Number of Columns Extracted,Number of Rows,Status
0,cutting_I.csv,cutting_I,7,1032,Successful
1,cutting_H.csv,cutting_H,7,1112,Successful
2,cutting_10.csv,cutting_10,7,1256,Successful
3,cutting_11.csv,cutting_11,7,1167,Successful
4,cutting_9.csv,cutting_9,7,1361,Successful
5,cutting_8.csv,cutting_8,7,1485,Successful
6,cutting_6.csv,cutting_6,7,1204,Successful
7,cutting_A.csv,cutting_A,7,1423,Successful
8,cutting_7.csv,cutting_7,7,1118,Successful
9,cutting_5.csv,cutting_5,7,1304,Successful


✅ The required columns have been extracted, and the data has been numerized and saved in the '../02_Processed_Data/02_Necessary_Columns' directory.
